# Ejercicio 11: Reflexión Final y Síntesis del Trabajo Práctico

En esta sección final respondemos a las preguntas integradoras del trabajo práctico, conectando la física del tomógrafo, el álgebra lineal y los métodos computacionales implementados a lo largo de los ejercicios.

### a) ¿Por qué la reconstrucción tomográfica se considera un problema inverso?

* **Problema directo vs. problema inverso:**
  * En un **problema directo**, conocemos el estado interno del objeto o paciente (la imagen $x$) y cómo opera el escáner (la matriz $A$). Con eso, calculamos fácilmente qué mediciones registrarán los sensores mediante una simple multiplicación: $b = Ax$.
  * En cambio, la **reconstrucción tomográfica es un problema inverso** porque en la práctica clínica ocurre exactamente lo opuesto: no podemos ver directamente el interior del paciente. Lo que tenemos son las mediciones externas de los detectores ($b$) y la geometría de los rayos ($A$), y nuestro objetivo es "ir hacia atrás", es decir, deducir la distribución interna de tejidos ($x$) a partir de los datos observados.

* **Variables del problema:**
  * **Información conocida:** El vector de mediciones del escáner $b \in \mathbb{R}^m$ y la matriz de proyección $A \in \mathbb{R}^{m \times n^2}$ (que describe qué píxeles atraviesa cada rayo).
  * **Variable que se busca recuperar:** El vector $x \in \mathbb{R}^{n^2}$, que representa los coeficientes de atenuación de cada píxel de la imagen que queremos reconstruir.

* **El papel del modelo $b = Ax$:**
  Es el puente matemático que conecta la física con la computación. Nace de aplicar logaritmo a la ley exponencial de Beer-Lambert ($-\ln(I/I_0) = \int \mu \, dr$). Esto linealiza el fenómeno y nos permite modelar cada medición del sensor como una suma discreta de atenuaciones a lo largo del rayo, formulando todo el proceso como un sistema de ecuaciones lineales estándar.

### b) ¿Cómo puede determinarse, a partir de $A$, si las mediciones permiten recuperar una única imagen?

* **Condición de unicidad (Rango y Espacio Nulo):**
  Para que una imagen pueda recuperarse de manera única, no pueden existir ambigüedades en el sistema. En términos de álgebra lineal, esto exige que el **espacio nulo de $A$ sea trivial**:
  $$\dim(\operatorname{Nul}(A)) = 0$$
  Por el Teorema de la Dimensión (Rango-Nulidad):
  $$\operatorname{rango}(A) + \dim(\operatorname{Nul}(A)) = n^2 = 64$$
  Por lo tanto, la reconstrucción es única si y solo si la matriz tiene rango completo de columnas: $\operatorname{rango}(A) = n^2 = 64$.

* **Lo que observamos en nuestro trabajo:**
  * Para $A_2$ (16 rayos): $\operatorname{rango}(A_2) = 15 \implies \dim(\operatorname{Nul}(A_2)) = 49$.
  * Para $A_4$ (44 rayos): $\operatorname{rango}(A_4) = 27 \implies \dim(\operatorname{Nul}(A_4)) = 37$.
  * Para $A_7$ (119 rayos): $\operatorname{rango}(A_7) = 43 \implies \dim(\operatorname{Nul}(A_7)) = 21$.
  
  ¡En ninguno de los tres casos la solución es única! Incluso con 119 mediciones para 64 incógnitas, muchos rayos son linealmente dependientes entre sí y el rango nunca llega a 64.

* **¿Qué ocurre si existe un vector $v \neq 0$ tal que $Av = 0$?**
  Si tenemos una imagen candidata $x$ que explica las mediciones ($Ax = b$), podemos construir otra imagen sumándole cualquier múltiplo de $v$:
  $$x_{\text{alt}} = x + \gamma v$$
  Al pasar por el escáner:
  $$A x_{\text{alt}} = A(x + \gamma v) = Ax + \gamma Av = b + 0 = b$$
  El tomógrafo genera **exactamente las mismas mediciones** para ambas imágenes. Físicamente, el vector $v$ representa un "patrón invisible" o artefacto fantasma (en el Ejercicio 4 vimos mediante SVD que tiene forma de damero con valores positivos y negativos que se compensan a lo largo de cada rayo).

### c) ¿Cómo afecta el ruido al sistema de ecuaciones?

* **Impacto del ruido (Incompatibilidad):**
  En condiciones reales, los detectores sufren ruido térmico y estadístico, por lo que medimos $b_\sigma = b + \text{ruido}$. Este ruido "empuja" al vector de mediciones fuera del espacio generado por las columnas de $A$:
  $$b_\sigma \notin \operatorname{Col}(A)$$
  Como consecuencia inmediata, el sistema lineal $Ax = b_\sigma$ se vuelve **incompatible**: no existe ninguna imagen en el mundo que cumpla exactamente todas las ecuaciones a la vez.

* **¿Qué problema resuelve mínimos cuadrados?**
  Como no hay solución exacta, mínimos cuadrados busca la "mejor aproximación posible": encuentra el vector $\hat{x}$ cuya predicción $A\hat{x}$ esté a la menor distancia euclídea posible de las mediciones ruidosas $b_\sigma$, minimizando la función de costo:
  $$E(x) = \frac{1}{2} \|Ax - b_\sigma\|_2^2$$

* **Interpretación del residuo:**
  El residuo $r = A\hat{x} - b_\sigma$ representa la discrepancia entre lo que nuestra imagen reconstruida proyecta en los detectores y lo que el escáner realmente registró. En el Ejercicio 7 demostramos geométricamente que este residuo es **perpendicular al espacio columna de $A$** ($A^T r = 0$), lo que confirma que $A\hat{x}$ es la proyección ortogonal de $b_\sigma$ sobre $\operatorname{Col}(A)$.

* **¿Es la norma del residuo suficiente para evaluar la calidad?**
  **No, no es suficiente.** La norma del residuo $\|A\hat{x} - b_\sigma\|$ solo mide qué tan bien coinciden las ecuaciones en el espacio de detectores, pero no nos dice nada de la calidad visual ni anatómica de la imagen.
  En el Ejercicio 6 vimos que a medida que aumenta el ruido, mínimos cuadrados logra hacer su residuo pequeño sobreajustando el ruido del sensor (*overfitting*). Sin embargo, la imagen resultante queda llena de artefactos granulares, con un error altísimo respecto al fantoma real e incluso con valores de píxeles negativos, que son físicamente imposibles para una densidad de tejido.

### d) ¿Qué supuesto sobre la imagen introduce el término $\|Dx\|^2$?

* **Supuesto físico incorporado:**
  La matriz $D = \begin{pmatrix} D_h \\ D_v \end{pmatrix}$ calcula las diferencias finitas entre píxeles vecinos contiguos (horizontales y verticales). Por lo tanto, el término $\|Dx\|^2 = \sum (x_i - x_j)^2$ mide la falta de suavidad o rugosidad espacial de la imagen.
  Al agregar este término al costo con un peso $\lambda$, introducimos el supuesto previo (*prior*) de que **los tejidos del cuerpo humano son en su mayoría continuos, suaves y homogéneos**, y que los saltos bruscos aleatorios píxel a píxel se deben al ruido y no a la anatomía del paciente.

* **Comportamiento al modificar $\lambda$ (el trade-off o compromiso):**
  * **Con $\lambda = 0$ (sin regularización):** Solo importa ajustar las mediciones. La imagen reconstruida queda muy ruidosa, con efecto "sal y pimienta" y píxeles negativos.
  * **Con $\lambda$ moderado (alrededor de 0.1 a 0.3):** Es el **balance óptimo**. La penalización suprime eficazmente las oscilaciones de alta frecuencia del ruido sin borrar las estructuras reales del objeto, logrando contornos nítidos y el menor error respecto al fantoma.
  * **Con $\lambda$ muy grande ($\ge 1.0$):** El costo le da un peso excesivo a la suavidad. La penalización aplasta todas las diferencias espaciales, borrando los bordes de los órganos internos y convirtiendo la imagen en una mancha borrosa gris y homogénea (sobre-suavizado).

### e) Comparación entre `np.linalg.lstsq`, Gradiente Descendiente y Newton

| Criterio | `np.linalg.lstsq` | Gradiente Descendiente | Método de Newton |
| :--- | :--- | :--- | :--- |
| **Tipo de método** | Directo (álgebra lineal) | Iterativo (primer orden) | Iterativo (segundo orden) |
| **Iteraciones** | 1 cálculo directo (basado en SVD) | Muchas (decenas o cientos de pasos) | **Exactamente 1 iteración** |
| **Parámetros requeridos** | Ninguno (solo $A$ y $b$) | Tasa de aprendizaje $\alpha$ y criterio de parada | Ninguno para el paso (resuelve $Hp = g$) |
| **Uso de memoria** | Moderado a alto | Mínimo (no almacena matrices densas) | Muy alto (requiere la matriz Hessiana $H$) |
| **Solución obtenida** | Solución de norma mínima | Solución idéntica a `lstsq` al converger | Solución idéntica a `lstsq` |

* **¿Por qué Newton alcanza la solución en una sola iteración?**
  El método de Newton se basa en aproximar la función de costo por su polinomio de Taylor de segundo orden en torno al punto actual:
  $$E(x + p) \approx E(x) + \nabla E(x)^T p + \frac{1}{2} p^T H p$$
  En nuestro problema, la función de costo $E(x) = \frac{1}{2}\|Ax - b\|^2$ **ya es exactamente un polinomio cuadrático puro**, y su Hessiana $H = A^T A$ es constante en todo el espacio. Por lo tanto, el desarrollo de Taylor de orden 2 no es una aproximación local, sino que es **idéntico a la función real en todo $\mathbb{R}^{n^2}$**.
  Al resolver el sistema $H p = \nabla E(x)$, el método encuentra de un solo salto el mínimo global exacto de la cuadrática.

### f) Síntesis Integradora del Trabajo Práctico

El problema de reconstrucción tomográfica nos permite articular de forma natural los grandes pilares del álgebra lineal y los métodos computacionales:

1. **El modelo $b = Ax$ (Modelado Físico):**
   Traduce el fenómeno continuo de absorción de rayos X (Ley de Beer-Lambert) en un sistema de ecuaciones lineales discreto. Cada fila representa la trayectoria de un rayo y cada columna un vóxel de tejido.

2. **El Rango y el Espacio Nulo (Unicidad y Geometría):**
   Nos permiten analizar si la geometría del tomógrafo es suficiente para recuperar la imagen. El rango mide cuánta información independiente capturan los rayos, mientras que el espacio nulo describe las regiones y patrones espaciales "ciegos" que el escáner no puede detectar.

3. **Mínimos Cuadrados y Proyecciones Ortogonales (Manejo del Ruido):**
   Explican qué hacer cuando los datos reales del sensor se vuelven incompatibles por ruido ($b_\sigma \notin \operatorname{Col}(A)$). La solución óptima es la proyección ortogonal del vector de mediciones sobre el espacio columna de $A$, garantizando un residuo perpendicular ($A^T r = 0$).

4. **Métodos de Optimización (Escalabilidad Numérica):**
   Muestran cómo resolver el sistema en problemas reales de alta resolución (como $128 \times 128$). En lugar de invertir matrices gigantescas que agotarían la memoria, usamos métodos iterativos como Gradiente Descendiente junto con matrices dispersas, controlando la convergencia a través de los autovalores de la Hessiana ($0 < \alpha < 2/\mu_{\max}$).

5. **Regularización y Principio de Morozov (Estabilidad y Calidad Clínica):**
   Resuelven la naturaleza mal condicionada del problema inverso. Al sumar un término de suavidad $\frac{\lambda}{2}\|Dx\|^2$, incorporamos conocimiento físico sobre la anatomía para filtrar el ruido, y el principio de Morozov nos da un criterio riguroso para calibrar $\lambda$ a ciegas en base al nivel de ruido de los detectores.